# Defining global variables

In [ ]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 6

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [ ]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

# Light-weight adaptation - Preparation

#### To do light-weight adaptation in Kaggle, we need the following packages

* `accelerate`: A library that simplifies running and training large models efficiently across GPUs/CPUs
* `peft`: The package to configurate LoRA
* `bitsandbytes`: The package to configurate quantization
* `trl`: A full stack package providing a set of tools to train transformer language models so we don't need to code everything, e.g., writing training loop, from scratch

Install and upgrade them by running the following cell

In [ ]:
!pip install --upgrade -q transformers peft accelerate bitsandbytes
!pip install -q trl

#### Model choice and load the tokenizer

We continue to work with Qwen/Qwen2.5-1.5B-Instruct ([**model card**](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)) as in the previous lecture

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Specify the device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Choose the model and load the tokenizer
BACKBONE = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    device_map = device
)

In [ ]:
prompt = 'What are people discussing in NVIDIA earnings call for the fiscal quarter 2025Q1?'

encoded_item = tokenizer(
    prompt,
    return_attention_mask = True,
    return_tensors = 'pt'
)

# Move the input and model to the device
model.to(device)
input_ids = encoded_item['input_ids'].to(device)
attention_mask = encoded_item['attention_mask'].to(device)

# Generate the new text
with torch.no_grad():
    generated_text = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        temperature = 1,
        max_new_tokens = 500,
        pad_token_id = tokenizer.eos_token_id
    )[0]

# Discard the prompt from the generated text
generated_text = generated_text[attention_mask.sum(dim = 1)[0]:]

# Decode
tokenizer.decode(generated_text, skip_special_tokens = True)

# Quantization

To do quantization, we need to use the class `BitsAndBytesConfig` in the `transformers` package. To learn about it, check [**this page**](https://huggingface.co/docs/transformers/en/quantization/bitsandbytes).

Important arguments include:
* `load_in_4bit`: (THIS IS THE KEY ARGUMENT) Set it to `True` and model weights will be loaded in 4-bits
* `load_in_8bit`: Set it to `True` and model weights will be loaded in 8-bits
* `bnb_4bit_quant_type`: This sets the quantization data type in the `bnb.nn.Linear4Bit` layers. Usually, set this argument to `nf4`
* `bnb_4bit_compute_dtype`: This sets the computational type which might be different than the input type 

Let's load the model in 4 bits.

In [ ]:
from transformers import BitsAndBytesConfig

# Configurate the quantization setting
quantization_config = BitsAndBytesConfig(
    ...
)

# Load the model and tell the model to quantize its weights
model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    device_map = device,
    quantization_config = ...,
)

# LoRA

To use LoRA, we need a package called `peft`. From it, import `LoraConfig` (which we can configurate LoRA), and `get_peft_model`, which we can adapt the model based on the LoRA configuration. To learn about it, check [**this page**](https://huggingface.co/docs/peft/package_reference/lora).

An important argument is `target_modules`. This takes a list of layer names (e.g., linear projection layers) where LoRA adapters will be inserted. To know which modules we want to apply LoRA, first inspect the model.

In [ ]:
model

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = ...,
    lora_alpha = ...,
    target_modules = ...
    lora_dropout = 0.05,
    bias = 'none',
    task_type = 'CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

## Prepare the data

We will fine-tune this model using NVIDIA's transcripts data. To this end, we need to organize the data into the desired format, which is a HuggingFace dataset.

You can think about the dataset as a list of dictionaries, each of which **must** have a value whose key is named `text`.

I wrote a function called `organize_dataset` to do this for you in the `Week_5.Code.data_preparation` script.

In [ ]:
import os
from Week_6.Code.data_preparation import organize_dataset

DATA_DIR = os.path.join(BASE_DIR, 'Data', 'Transcripts')
finetuned_dataset = organize_dataset(DATA_DIR)
finetuned_dataset[0]

## Start fine-tuning

To prepare for fine-tuning, we need to prepare training arguments, which regulate how the training works. To do that, we use the `TrainingArguments` class in the `transformers` package.

Important arguments in the `TrainingArguments` include:
* `output_dir`: where to store the model after fine-tuning
* `per_device_train_batch_size`: the batch size
* `learning_rate`: the learning rate (for the stochastic gradient descent)
* `num_train_epochs`: the number of fine-tuning iterations. For example, `num_train_epochs = 2` means the model reads the training data twice.
* `logging_steps`: the number of steps after which the training progress is reported (or logged)
* `save_strategy`: governs how the fine-tuned model is saved.
    - `'no'`: No save is done during training
    - `'epoch'`: Save is done at the end of each epoch
    - `'steps'`: Save is done every save_steps
    - `'best'`: Save is done whenever a new best_metric is achieved

Finally, after having everything---the model, the data, the training arguments---we input it into a class called `SFTTrainer` (SFT: **S**upervised **F**ine-**T**uning).

Let's do it!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = '/kaggle/working/model',
    per_device_train_batch_size = ...,
    gradient_accumulation_steps = 5,    
    learning_rate = ...,
    num_train_epochs = ...,
    logging_steps = ...,
    save_strategy = ...
)

trainer = SFTTrainer(
    model = ...,
    train_dataset = ...,
    args = ...
)

trainer.train()                         # Train, or fine-tune
trainer.save_model(                     # Save model after training
    '/kaggle/working/model'
)

# Check

In [ ]:
prompt = 'What are people discussing in NVIDIA earnings call for the fiscal quarter 2025Q1?'

encoded_item = tokenizer(
    prompt,
    return_attention_mask = True,
    return_tensors = 'pt'
)

# Move the input and model to the device
model.to(device)
input_ids = encoded_item['input_ids'].to(device)
attention_mask = encoded_item['attention_mask'].to(device)

# Generate the new text
with torch.no_grad():
    generated_text = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        temperature = 1,
        max_new_tokens = 500,
        pad_token_id = tokenizer.eos_token_id
    )[0]

# Discard the prompt from the generated text
generated_text = generated_text[attention_mask.sum(dim = 1)[0]:]

# Decode
tokenizer.decode(generated_text, skip_special_tokens = True)

# Retrieval-Augmented Generation

The implementation of RAG can be found [**here**](https://www.kaggle.com/code/shinomoriaoshi/textual-analysis-in-finance-week-6-rag).